# Solutions · 02 · Stiffness and choosing the integrator

Worked solutions to the exercises of [notebook 02](../notebooks/02_stiffness_and_choosing_the_integrator.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, stiffness, grids, boundary as bc, operators as op, stencils as st
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Repeat the integrator comparison for the advection equation `problems.advection_periodic(scheme='centred')` whose spectrum is imaginary. Which explicit methods are stable at all, and what does BDF2 do to the pulse's amplitude after one period?

In [2]:
p = pr.advection_periodic(scheme="centred")
lam = stiffness.spectrum(p.rhs, p.y0)
for m in ("euler", "heun", "rk3", "rk4"):
    print(f"{m:<5}: step limit {stiffness.explicit_step_limit(lam, m):.2e}")
s = ti.bdf(p.rhs, p.y0, (0, p.t_end), 0.005, 2, band=p.band, n_out=1)
print(f"BDF2 after one period: amplitude {s.y[-1].max():.3f} of the initial 1.000 - the A-stable method damps undamped waves")

euler: step limit 7.08e-09
heun : step limit 8.41e-06
rk3  : step limit 8.66e-03
rk4  : step limit 1.41e-02


BDF2 after one period: amplitude 0.904 of the initial 1.000 - the A-stable method damps undamped waves


Euler and Heun have no stable step at all on a purely imaginary spectrum; RK3 and RK4 do (their regions cross
the imaginary axis). BDF2, A-stable, is stable at any step - and damps the pulse by a tenth in one period,
because its |R| < 1 off the real axis: stability is not fidelity for non-dissipative problems.

## Exercise 2

Start Crank-Nicolson (the trapezoidal rule) on the hot band with two backward-Euler half steps (Rannacher's trick) and compare the ringing with the plain start.

In [3]:
g = grids.uniform(0, 1, 40)
f = lambda t, u: op.diffusion(u, g, 1.0, bc.Dirichlet(0.0), bc.Dirichlet(0.0), t)   # noqa: E731
u0 = np.where((g.x > 0.4) & (g.x < 0.6), 1.0, 0.0); h = 0.02
plain = ti.trapezoidal(f, u0, (0, 0.1), h, band=(1, 1), n_out=5)
start = ti.bdf(f, u0, (0, h), h / 2, 1, band=(1, 1), n_out=1).y[-1]           # two backward-Euler half steps
rann = ti.trapezoidal(f, start, (h, 0.1), h, band=(1, 1), n_out=4)
A = st.diff_operator(g.x, 2, 2).to_dense(); A[0, :] = A[-1, :] = 0.0
ref = ti.exponential(A, u0, (0, 0.1)).y[-1]
print(f"plain trapezoidal: min value {plain.y[-1].min():+.4f}, error {np.max(np.abs(plain.y[-1] - ref)):.2e}")
print(f"Rannacher start:   min value {rann.y[-1].min():+.4f}, error {np.max(np.abs(rann.y[-1] - ref)):.2e}")

plain trapezoidal: min value -0.1525, error 2.91e-01
Rannacher start:   min value -0.0000, error 2.53e-03


Two backward-Euler half steps damp the sawtooth modes that the trapezoidal rule would carry forever; after
them Crank-Nicolson proceeds at second order with no ringing. The error against the exact semi-discrete
solution drops by an order of magnitude.

## Exercise 3

**Implement it yourself:** write the trapezoidal rule for the linear semi-discrete heat equation with one factorisation of (I - hA/2) and reproduce the ringing figure; then halve h until the ringing disappears and relate that h to |R(h lambda_max)|.

In [4]:
lam_max = np.linalg.eigvalsh(A[1:-1, 1:-1]).min()
for h in (0.02, 0.01, 0.005, 0.0025, 0.00125):
    M = np.linalg.inv(np.eye(41) - h * A / 2) @ (np.eye(41) + h * A / 2)     # the trapezoidal one-step matrix (dense here for clarity)
    u = u0.copy()
    for _ in range(int(round(0.1 / h))):
        u = M @ u
    R = (1 + h * lam_max / 2) / (1 - h * lam_max / 2)
    print(f"h = {h:7.5f}: |R(h lambda_max)| = {abs(R):.3f}, min value {u.min():+.4f}, error {np.max(np.abs(u - ref)):.1e}")

h = 0.02000: |R(h lambda_max)| = 0.969, min value -0.1525, error 2.9e-01
h = 0.01000: |R(h lambda_max)| = 0.939, min value -0.0087, error 1.3e-01
h = 0.00500: |R(h lambda_max)| = 0.882, min value +0.0000, error 1.3e-02
h = 0.00250: |R(h lambda_max)| = 0.777, min value +0.0000, error 1.1e-05
h = 0.00125: |R(h lambda_max)| = 0.600, min value +0.0000, error 2.0e-06


The negative values disappear once |R(hλ_max)| is comfortably below 1 - at h = 0.005, where the factor is
0.88 - and the error against the exact semi-discrete solution reaches its second-order regime a step later;
this is the step at which the fastest modes are resolved rather than merely stable. With a Rannacher start
one can keep the large step instead.